<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания: 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

<b> Описание задачи: </b>

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

<b> Требования к базовому классу PaymentMethod: </b>

<b> • Атрибуты: </b> ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).

<b> • Методы: </b>

- ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.

- CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.

- GetPaymentDetails(): метод для получения деталей способа оплаты.

<b> Требования к производным классам: </b>

1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Интерфейсы
public interface ITransactionLogger
{
    void LogTransaction(string transactionDetails);
    string GetTransactionHistory();
    void ClearHistory();
}

public interface ISecurityValidator
{
    bool ValidateSecurity();
    void SetSecurityLevel(int level);
    int GetSecurityScore();
}

public interface ICurrencyConverter
{
    decimal ConvertAmount(decimal amount, string targetCurrency);
    string DefaultCurrency { get; set; }
    decimal GetExchangeRate(string fromCurrency, string toCurrency);
}

// Базовая реализация логгера для управления зависимостями
public class FileTransactionLogger : ITransactionLogger
{
    private List<string> _transactionHistory = new List<string>();
    private string _loggerName;

    public FileTransactionLogger(string name = "FileLogger")
    {
        _loggerName = name;
    }

    public void LogTransaction(string transactionDetails)
    {
        string logEntry = $"{DateTime.Now:dd.MM.yyyy HH:mm:ss} - [{_loggerName}] {transactionDetails}";
        _transactionHistory.Add(logEntry);
        Console.WriteLine($"Записано в лог: {transactionDetails}");
    }

    public string GetTransactionHistory()
    {
        return _transactionHistory.Count == 0 
            ? "История пуста" 
            : string.Join("\n", _transactionHistory);
    }

    public void ClearHistory()
    {
        _transactionHistory.Clear();
        Console.WriteLine("История транзакций очищена");
    }
}

// Сервис для управления зависимостями
public class PaymentService
{
    private readonly ITransactionLogger _logger;
    private readonly ICurrencyConverter _converter;

    public PaymentService(ITransactionLogger logger, ICurrencyConverter converter = null)
    {
        _logger = logger;
        _converter = converter;
    }

    public void ProcessPaymentWithLogging(PaymentMethod payment, decimal amount, string description = "")
    {
        _logger.LogTransaction($"Начало обработки: {payment.MethodName} - {amount} RUB");
        payment.ProcessPaymentWithCheck(amount);
        _logger.LogTransaction($"Завершение обработки: {payment.MethodName}");
    }

    public decimal ConvertIfNeeded(PaymentMethod payment, decimal amount, string targetCurrency)
    {
        if (_converter != null && payment.Currency != targetCurrency)
        {
            return _converter.ConvertAmount(amount, targetCurrency);
        }
        return amount;
    }
}

public abstract class PaymentMethod : ITransactionLogger, ISecurityValidator
{
    // Существующие атрибуты
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }
    public DateTime CreatedDate { get; set; }
    public bool IsActive { get; set; }
    public string Currency { get; set; }
    public int SecurityLevel { get; set; }
    protected List<string> TransactionHistory { get; set; }
    public string Category { get; set; }
    public int Priority { get; set; }
    public decimal SuccessRate { get; set; }
    private int _todayTransactionCount;

    // НОВЫЕ АТРИБУТЫ
    public string ProviderName { get; set; }
    public DateTime LastUsed { get; set; }
    public int MaxDailyTransactions { get; set; }
    public string CountryCode { get; set; }
    public decimal Balance { get; set; }

    protected PaymentMethod() 
    {
        CreatedDate = DateTime.Now;
        IsActive = true;
        Currency = "RUB";
        SecurityLevel = 1;
        TransactionHistory = new List<string>();
        Category = "Общие";
        Priority = 1;
        SuccessRate = 95.0m;
        _todayTransactionCount = 0;
        
        // Инициализация новых атрибутов
        ProviderName = "Неизвестный провайдер";
        LastUsed = DateTime.MinValue;
        MaxDailyTransactions = 100;
        CountryCode = "RU";
        Balance = 0;
    }
    
    protected PaymentMethod(int id, string name, decimal minAmount) : this()
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
    }

    // Существующие методы
    public virtual void ProcessPayment(decimal amount)
    {
        if (_todayTransactionCount >= MaxDailyTransactions)
        {
            Console.WriteLine("Достигнут лимит daily транзакций");
            return;
        }

        _todayTransactionCount++;
        LastUsed = DateTime.Now;
        Console.WriteLine($"Транзакция через {MethodName}: {amount} {Currency}");
        LogTransaction($"Транзакция: {amount} {Currency}");
        
        if (new Random().Next(100) < SuccessRate)
        {
            Console.WriteLine("Транзакция успешна");
            Balance -= amount;
        }
        else
        {
            Console.WriteLine("Транзакция не удалась");
        }
    }

    // НОВЫЕ МЕТОДЫ
    public virtual void ReplenishBalance(decimal amount)
    {
        Balance += amount;
        LogTransaction($"Пополнение баланса: +{amount} {Currency}");
        Console.WriteLine($"Баланс пополнен на {amount}. Текущий баланс: {Balance}");
    }

    public virtual bool CanProcessPayment(decimal amount)
    {
        return IsActive && 
               Balance >= amount && 
               amount >= MinAmount && 
               _todayTransactionCount < MaxDailyTransactions &&
               ValidateSecurity();
    }

    public virtual string GetProviderInfo()
    {
        return $"Провайдер: {ProviderName}, Страна: {CountryCode}, Активен: {(IsActive ? "Да" : "Нет")}";
    }

    public virtual void ResetDailyCounter()
    {
        _todayTransactionCount = 0;
        Console.WriteLine("Счетчик daily транзакций сброшен");
    }

    // Явная реализация интерфейса ITransactionLogger
    void ITransactionLogger.LogTransaction(string transactionDetails)
    {
        string logEntry = $"{DateTime.Now:dd.MM.yyyy HH:mm:ss} - [Явная реализация] {transactionDetails}";
        TransactionHistory.Add(logEntry);
    }

    string ITransactionLogger.GetTransactionHistory()
    {
        return TransactionHistory.Count == 0 
            ? "История транзакций пуста (явная реализация)" 
            : $"Явная реализация логгера:\n{string.Join("\n", TransactionHistory)}";
    }

    // Обычная реализация интерфейса ITransactionLogger
    public virtual void LogTransaction(string transactionDetails)
    {
        string logEntry = $"{DateTime.Now:dd.MM.yyyy HH:mm:ss} - {transactionDetails}";
        TransactionHistory.Add(logEntry);
    }

    public virtual string GetTransactionHistory()
    {
        return TransactionHistory.Count == 0 
            ? "История пуста" 
            : string.Join("\n", TransactionHistory);
    }

    public virtual void ClearHistory()
    {
        TransactionHistory.Clear();
        Console.WriteLine("История транзакций очищена");
    }

    // Реализация ISecurityValidator
    public virtual bool ValidateSecurity()
    {
        return SecurityLevel >= 1 && IsActive && !string.IsNullOrEmpty(ProviderName);
    }

    public virtual void SetSecurityLevel(int level)
    {
        SecurityLevel = level;
        Console.WriteLine($"Уровень безопасности установлен: {level}");
    }

    public virtual int GetSecurityScore()
    {
        int score = SecurityLevel * 10;
        score += IsActive ? 20 : 0;
        score += !string.IsNullOrEmpty(ProviderName) ? 15 : 0;
        return score;
    }

    // Остальные существующие методы
    public virtual bool CheckMinimumAmount(decimal amount)
    {
        if (amount >= MinAmount)
        {
            Console.WriteLine("Средств достаточно");
            return true;
        }
        else
        {
            Console.WriteLine("Недостаточно средств");
            return false;
        }
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID: {PaymentMethodId}, Способ: {MethodName}, Мин. сумма: {MinAmount} {Currency}");
        Console.WriteLine($"Баланс: {Balance}, Провайдер: {ProviderName}");
    }

    public virtual void ProcessPaymentWithCheck(decimal amount)
    {
        if (CanProcessPayment(amount))
        {
            ProcessPayment(amount);
        }
        else
        {
            Console.WriteLine("Транзакция отменена - проверка не пройдена");
        }
    }

    public virtual void UpdatePriority(int newPriority)
    {
        Priority = newPriority;
        Console.WriteLine($"Приоритет изменен на: {newPriority}");
    }

    public virtual string GetStatus()
    {
        return $"{MethodName}: {(IsActive ? "Активен" : "Неактивен")}, Приоритет: {Priority}, Баланс: {Balance}";
    }
}

class OnlinePayment : PaymentMethod, ICurrencyConverter
{
    // Существующие атрибуты
    public string PaymentUrl { get; set; }
    public string DefaultCurrency { get; set; }
    public string GatewayProvider { get; set; }
    public int TimeoutSeconds { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public bool SupportsRecurring { get; set; }
    public string ApiVersion { get; set; }
    public int RetryCount { get; set; }
    public string EncryptionType { get; set; }
    
    public OnlinePayment() : base() 
    {
        DefaultCurrency = "RUB";
        GatewayProvider = "DefaultGateway";
        TimeoutSeconds = 30;
        
        // Инициализация новых атрибутов
        SupportsRecurring = false;
        ApiVersion = "1.0";
        RetryCount = 3;
        EncryptionType = "SSL";
        ProviderName = "OnlinePaymentProvider";
    }
    
    public OnlinePayment(string paymentUrl) : base(1, "Онлайн", 100m)
    {
        PaymentUrl = paymentUrl;
        DefaultCurrency = "RUB";
        ProviderName = "OnlinePaymentProvider";
    }
    
    // НОВЫЕ МЕТОДЫ
    public void SetupRecurringPayment(decimal amount, int intervalDays)
    {
        if (SupportsRecurring)
        {
            Console.WriteLine($"Настройка recurring платежа: {amount} каждые {intervalDays} дней");
            LogTransaction($"Recurring платеж настроен: {amount} каждые {intervalDays} дней");
        }
        else
        {
            Console.WriteLine("Recurring платежи не поддерживаются");
        }
    }

    public void UpdateApiVersion(string newVersion)
    {
        ApiVersion = newVersion;
        Console.WriteLine($"Версия API обновлена: {newVersion}");
    }

    public override string GetProviderInfo()
    {
        return base.GetProviderInfo() + $", API: {ApiVersion}, Шифрование: {EncryptionType}";
    }

    // Существующие методы
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Онлайн платеж через {GatewayProvider}");
        base.ProcessPayment(amount);
        Console.WriteLine($"URL: {PaymentUrl}, Retry: {RetryCount}");
    }
    
    public override bool CanProcessPayment(decimal amount)
    {
        return base.CanProcessPayment(amount) && 
               !string.IsNullOrEmpty(PaymentUrl) &&
               TimeoutSeconds > 0;
    }

    // Реализация ICurrencyConverter
    public decimal ConvertAmount(decimal amount, string targetCurrency)
    {
        decimal rate = GetExchangeRate(Currency, targetCurrency);
        decimal convertedAmount = amount * rate;
        Console.WriteLine($"Конвертация: {amount} {Currency} → {convertedAmount} {targetCurrency}");
        return convertedAmount;
    }

    public decimal GetExchangeRate(string fromCurrency, string toCurrency)
    {
        // Упрощенная логика конвертации
        var rates = new Dictionary<string, decimal>
        {
            {"RUB_USD", 0.011m},
            {"RUB_EUR", 0.010m},
            {"USD_RUB", 91.5m},
            {"EUR_RUB", 99.3m}
        };

        string key = $"{fromCurrency}_{toCurrency}";
        return rates.ContainsKey(key) ? rates[key] : 1.0m;
    }
}

class BankTransfer : PaymentMethod
{
    // Существующие атрибуты
    public decimal BankFee { get; set; }
    public string BankCode { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public string SwiftCode { get; set; }
    public string AccountNumber { get; set; }
    public string BankAddress { get; set; }
    public bool InternationalEnabled { get; set; }
    
    public BankTransfer() : base() 
    {
        BankCode = "044525";
        
        // Инициализация новых атрибутов
        SwiftCode = "";
        AccountNumber = "Не указан";
        BankAddress = "Не указан";
        InternationalEnabled = false;
        ProviderName = "BankTransferSystem";
    }
    
    public BankTransfer(decimal bankFee) : base(2, "Банковский перевод", 1000m)
    {
        BankFee = bankFee;
        ProviderName = "BankTransferSystem";
    }
    
    // НОВЫЕ МЕТОДЫ
    public void UpdateBankDetails(string accountNumber, string bankAddress)
    {
        AccountNumber = accountNumber;
        BankAddress = bankAddress;
        Console.WriteLine("Банковские реквизиты обновлены");
    }

    public void EnableInternationalTransfers(string swiftCode)
    {
        SwiftCode = swiftCode;
        InternationalEnabled = true;
        Console.WriteLine($"Международные переводы активированы. SWIFT: {swiftCode}");
    }

    public override string GetProviderInfo()
    {
        return base.GetProviderInfo() + $", SWIFT: {SwiftCode}, Международные: {(InternationalEnabled ? "Да" : "Нет")}";
    }

    // Существующие методы
    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal totalAmount = amount + BankFee;
        if (totalAmount >= MinAmount)
        {
            Console.WriteLine($"Средств достаточно (комиссия: {BankFee} {Currency})");
            return true;
        }
        else
        {
            Console.WriteLine($"Недостаточно средств (требуется: {MinAmount} {Currency})");
            return false;
        }
    }
    
    public void ProcessInternationalTransfer(decimal amount, string targetCountry)
    {
        if (InternationalEnabled)
        {
            Console.WriteLine($"Международный перевод в {targetCountry} через SWIFT: {SwiftCode}");
            ProcessPayment(amount);
        }
        else
        {
            Console.WriteLine("Международные переводы не активированы");
        }
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Комиссия: {BankFee} {Currency}, БИК: {BankCode}, Счет: {AccountNumber}");
    }
}

class CashPayment : PaymentMethod
{
    // Существующие атрибуты
    public string CashPickupPoint { get; set; }
    public string LocationAddress { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public string OperatingHours { get; set; }
    public string ContactPhone { get; set; }
    public decimal MaxCashAmount { get; set; }
    public bool HasATM { get; set; }
    
    public CashPayment() : base() 
    {
        LocationAddress = "Не указан";
        
        // Инициализация новых атрибутов
        OperatingHours = "09:00-18:00";
        ContactPhone = "Не указан";
        MaxCashAmount = 100000;
        HasATM = false;
        ProviderName = "CashNetwork";
    }
    
    public CashPayment(string cashPickupPoint) : base(3, "Наличные", 150m)
    {
        CashPickupPoint = cashPickupPoint;
        ProviderName = "CashNetwork";
    }
    
    // НОВЫЕ МЕТОДЫ
    public void UpdateLocationInfo(string address, string hours, string phone)
    {
        LocationAddress = address;
        OperatingHours = hours;
        ContactPhone = phone;
        Console.WriteLine("Информация о локации обновлена");
    }

    public string GetLocationInfo()
    {
        return $"Адрес: {LocationAddress}, Часы работы: {OperatingHours}, Телефон: {ContactPhone}";
    }

    public override bool CanProcessPayment(decimal amount)
    {
        return base.CanProcessPayment(amount) && amount <= MaxCashAmount;
    }

    // Существующие методы
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Обработка наличных в точке: {CashPickupPoint}");
        base.ProcessPayment(amount);
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Место выдачи: {CashPickupPoint}, Адрес: {LocationAddress}, Макс. сумма: {MaxCashAmount}");
    }
}

class CreditCardPayment : PaymentMethod, ICurrencyConverter
{
    // Существующие атрибуты
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }
    public string DefaultCurrency { get; set; }
    public string CardType { get; set; }
    
    // НОВЫЕ АТРИБУТЫ
    public DateTime ExpiryDate { get; set; }
    public string CVV { get; set; }
    public decimal CreditLimit { get; set; }
    public decimal AvailableCredit { get; set; }
    
    public CreditCardPayment() : base() 
    {
        DefaultCurrency = "RUB";
        CardType = "Visa";
        
        // Инициализация новых атрибутов
        ExpiryDate = DateTime.Now.AddYears(3);
        CVV = "000";
        CreditLimit = 100000;
        AvailableCredit = 100000;
        ProviderName = "CardProcessor";
    }
    
    public CreditCardPayment(string cardNumber, string cardHolder) : base(4, "Кредитная карта", 50m)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
        ProviderName = "CardProcessor";
    }
    
    // НОВЫЕ МЕТОДЫ
    public bool IsCardExpired()
    {
        return ExpiryDate < DateTime.Now;
    }

    public void UpdateCreditLimit(decimal newLimit)
    {
        CreditLimit = newLimit;
        AvailableCredit = newLimit - (CreditLimit - AvailableCredit);
        Console.WriteLine($"Кредитный лимит обновлен: {newLimit}");
    }

    public override bool CanProcessPayment(decimal amount)
    {
        return base.CanProcessPayment(amount) && 
               !IsCardExpired() && 
               AvailableCredit >= amount;
    }

    public override void ProcessPayment(decimal amount)
    {
        if (!IsCardExpired())
        {
            AvailableCredit -= amount;
            Console.WriteLine($"Обработка {CardType} карты. Доступный кредит: {AvailableCredit}");
            base.ProcessPayment(amount);
            Console.WriteLine($"Карта: ****{CardNumber.Substring(CardNumber.Length - 4)}");
        }
        else
        {
            Console.WriteLine("Карта просрочена - транзакция отклонена");
        }
    }

    // Реализация ICurrencyConverter
    public decimal ConvertAmount(decimal amount, string targetCurrency)
    {
        decimal rate = GetExchangeRate(Currency, targetCurrency);
        decimal convertedAmount = amount * rate;
        decimal conversionFee = convertedAmount * 0.02m; // 2% комиссия
        decimal totalAmount = convertedAmount + conversionFee;
        
        Console.WriteLine($"Конвертация по карте: {amount} {Currency} → {totalAmount} {targetCurrency} (включая комиссию 2%)");
        return totalAmount;
    }

    public decimal GetExchangeRate(string fromCurrency, string toCurrency)
    {
        var rates = new Dictionary<string, decimal>
        {
            {"RUB_USD", 0.011m},
            {"RUB_EUR", 0.010m},
            {"USD_RUB", 91.5m},
            {"EUR_RUB", 99.3m}
        };

        string key = $"{fromCurrency}_{toCurrency}";
        return rates.ContainsKey(key) ? rates[key] : 1.0m;
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Держатель: {CardHolder}, Тип: {CardType}, Доступный кредит: {AvailableCredit}");
    }
}

// Демонстрация работы
Console.WriteLine("=== РАСШИРЕННАЯ ДЕМОНСТРАЦИЯ СИСТЕМЫ ПЛАТЕЖЕЙ ===\n");

// Создание экземпляров с новыми атрибутами
var onlinePayment = new OnlinePayment("https://payment-gateway.com/process")
{
    MinAmount = 100,
    GatewayProvider = "CloudPayments",
    SupportsRecurring = true,
    ApiVersion = "2.1",
    Balance = 5000
};

var bankTransfer = new BankTransfer(120)
{
    MinAmount = 1000,
    Balance = 15000,
    AccountNumber = "40702810123456789012"
};

var cashPayment = new CashPayment("Сбербанк на Ленина")
{
    MinAmount = 150,
    Balance = 10000,
    MaxCashAmount = 50000,
    LocationAddress = "ул. Ленина, д. 25"
};

var creditCard = new CreditCardPayment("4111111111111111", "Иванов И.И.")
{
    MinAmount = 50,
    CardType = "MasterCard",
    Balance = 8000,
    ExpiryDate = DateTime.Now.AddYears(2)
};

Console.WriteLine("=== ДЕМОНСТРАЦИЯ НОВЫХ АТРИБУТОВ И МЕТОДОВ ===");
onlinePayment.ReplenishBalance(2000);
bankTransfer.EnableInternationalTransfers("SABRRUMM");
cashPayment.UpdateLocationInfo("ул. Ленина, д. 25", "09:00-20:00", "+7-999-123-45-67");
creditCard.UpdateCreditLimit(150000);

Console.WriteLine("\n=== ИНФОРМАЦИЯ О ПРОВАЙДЕРАХ ===");
Console.WriteLine(onlinePayment.GetProviderInfo());
Console.WriteLine(bankTransfer.GetProviderInfo());
Console.WriteLine(cashPayment.GetProviderInfo());
Console.WriteLine(creditCard.GetProviderInfo());

Console.WriteLine("\n=== ПРОВЕРКА ВОЗМОЖНОСТИ ПЛАТЕЖЕЙ ===");
Console.WriteLine($"Онлайн платеж возможен: {onlinePayment.CanProcessPayment(300)}");
Console.WriteLine($"Банковский перевод возможен: {bankTransfer.CanProcessPayment(1200)}");
Console.WriteLine($"Наличные возможны: {cashPayment.CanProcessPayment(200)}");
Console.WriteLine($"Кредитная карта возможна: {creditCard.CanProcessPayment(100)}");

Console.WriteLine("\n=== УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ ===");
var fileLogger = new FileTransactionLogger("MainLogger");
var paymentService = new PaymentService(fileLogger, onlinePayment);

paymentService.ProcessPaymentWithLogging(onlinePayment, 500, "Тестовый платеж");
paymentService.ProcessPaymentWithLogging(creditCard, 300, "Покупка в магазине");

Console.WriteLine("\n=== ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСА ===");
ITransactionLogger explicitLogger = onlinePayment;
explicitLogger.LogTransaction("Тест явной реализации");
Console.WriteLine(explicitLogger.GetTransactionHistory());

Console.WriteLine("\n=== ОЦЕНКИ БЕЗОПАСНОСТИ ===");
Console.WriteLine($"Оценка безопасности онлайн платежа: {onlinePayment.GetSecurityScore()}");
Console.WriteLine($"Оценка безопасности кредитной карты: {creditCard.GetSecurityScore()}");

Console.WriteLine("\n=== GENERIC КОЛЛЕКЦИИ С НОВЫМИ ВОЗМОЖНОСТЯМИ ===");
var onlineCollection = new PaymentCollection<OnlinePayment>();
onlineCollection.AddPayment(onlinePayment);

var backupOnline = new OnlinePayment("https://backup.com") 
{ 
    MethodName = "Резервный",
    Balance = 3000,
    ProviderName = "BackupProvider"
};
onlineCollection.AddPayment(backupOnline);

onlineCollection.ProcessAll(500);

Console.WriteLine("\n=== РАБОТА С RECURRING ПЛАТЕЖАМИ ===");
onlinePayment.SetupRecurringPayment(1000, 30);

Console.WriteLine("\n=== МЕЖДУНАРОДНЫЕ ПЕРЕВОДЫ ===");
bankTransfer.ProcessInternationalTransfer(1500, "США");

Console.WriteLine("\n=== ИНФОРМАЦИЯ О ЛОКАЦИЯХ ===");
Console.WriteLine(cashPayment.GetLocationInfo());

Console.WriteLine("\n=== ПРОВЕРКА СРОКА ДЕЙСТВИЯ КАРТЫ ===");
Console.WriteLine($"Карта просрочена: {creditCard.IsCardExpired()}");

Console.WriteLine("\n=== ФИНАЛЬНЫЕ СТАТУСЫ ===");
Console.WriteLine(onlinePayment.GetStatus());
Console.WriteLine(bankTransfer.GetStatus());
Console.WriteLine(cashPayment.GetStatus());
Console.WriteLine(creditCard.GetStatus());

Error: (741,87): error CS1002: ; expected
(743,20): error CS1002: ; expected
(743,20): error CS1513: } expected
(744,38): error CS1002: ; expected
(744,38): error CS1513: } expected
(745,27): error CS1002: ; expected
(748,51): error CS1002: ; expected
(750,21): error CS1002: ; expected
(750,21): error CS1513: } expected
(751,28): error CS1002: ; expected
(754,66): error CS1002: ; expected
(756,20): error CS1002: ; expected
(756,20): error CS1513: } expected
(757,28): error CS1002: ; expected
(760,84): error CS1002: ; expected
(762,19): error CS1002: ; expected
(762,19): error CS1513: } expected
(763,28): error CS1002: ; expected
(763,28): error CS1513: } expected
(764,28): error CS1002: ; expected
(812,79): error CS1026: ) expected
(812,79): error CS1002: ; expected
(812,106): error CS1002: ; expected
(812,107): error CS7017: Member definition, statement, or end-of-file expected